In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder , OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

# Loading the dataset to dataframe

Q1_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(Q1_path)

In [ ]:
# Task 2: Write your code here:

# displaying the first few rows
print(f"Dataset shape: {df.shape}") #just checking the shape
df.head()

In [ ]:
# Task 3: Write your code here:

# Checking data types and structure
df.info()

In [ ]:
# Task 4: Write your code here:

# Descriptive statistics for numerical columns
df.describe()

In [ ]:
# Task 5: Write your code here:

# delivery time distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery_Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:

#Drop the 'Order_ID' column from the data
df.drop('Order_ID', axis=1 , inplace = True) # we can save changes inplace :)
df.head()#just checking if col was removed


In [ ]:
# Task 2: Write your code here:

"""
my inital notes:
Detect with isnull(), isnull().sum()
Analyze missing percentage per column
Handle using dropna(subset=...) or fillna()
"""

#Handle missing values appropriately
# Missing values
print("Missing values:")
print(df.isnull().sum())

# EXtra way to Analyze missing values ( by percentage)
missing_percentage = (df.isnull().sum() / len(df)) * 100 # a series where (index -> dataset columns , entries -> missing percentages)
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

#print("Missing Data Analysis:")
missing_data.head(5) #cuz appearntly ,, only 5 attributes have missing entries (will display if its the last line)

# Drop rows where target (Delivery_Time) or key features are missing
df_clean = df.dropna(subset=['Delivery_Time', 'Weather', 'Time_of_Day']) #im not gonna do inplace here cuz i feel like i wanna keep the semi-original df somewhere
#also ,, im assuming the weather and Time_of_Day and Experience_yrs are key features (cuz Rain may cause delay , Night may cause difficulty to see the road..)



#im gonna infer the Traffic level & the driver experience, just to make a use of fillNa :D
df_clean['Traffic_Level'] = df_clean['Traffic_Level'].fillna(df_clean['Traffic_Level'].mode()[0]) #we access by index cuz there could be tie results
df_clean['Courier_Experience_yrs'] = df_clean['Courier_Experience_yrs'].fillna(df_clean['Courier_Experience_yrs'].mode()[0]) #we access by index cuz there could be tie results

#recheck missing values
# Missing values
print("\nMissing values:")
print(df_clean.isnull().sum())


In [ ]:
# Task 3: Write your code here:

#Check and remove duplicates if any exist
# 4. Do we have duplicate samples?

 #---doing it as a function cuz why not...
def check_duplicates(df):
  duplicates = df_clean.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_clean.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:

# Encode Categorical Features - one hot
onehot_encoder = OneHotEncoder(sparse_output=False)

for col in df_clean.select_dtypes(include=["object"]).columns:
    df_clean[col] = onehot_encoder.fit_transform(df_clean[col])

print('\nData after encoding:\n', df_clean) #show after encoding



In [ ]:
# Task 5: Write your code here:

#Apply feature scaling for all features (Use StandardScaler)
#----note!!! we should scale AFTER splitting train/test, to avoid data leakage,,,, but ill just stick to the exam convetion---

# Scale features - fit on train, transform both
scaler = StandardScaler()
features = df.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET
df_clean[features] = scaler.fit_transform(df_clean[features])

pd.DataFrame(df_clean, columns=df_clean.columns).head(3)

In [ ]:
# Task 6: Write your code here:

#Check for target imbalance and state if it is imbalanced or not

#class imbalance is a classification concept NOT regression!

In [ ]:
# Task 1: Write your code here:

X = df_clean.drop(columns=['Delivery_Time'])
y = df_clean['Delivery_Time']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train: {y_train.shape}, Test: {y_test.shape}")
X_train.head()
y_train.head()

In [ ]:
# Task 2,3,4,5: Write your code here:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

model = RandomForestClassifier(
      n_estimators=320,  # Number of trees
      max_depth=4
  ),

mae_scores = []

for train_idx, val_idx in kfold.split(X_train):
    X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx] #i guess i have to convert the dataframe to numpy array but i dont have the documintation :(
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))

mae_scores = np.array(mae_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")


In [ ]:
# Task 1: Write your code here:
# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
# Plot for Linear Regression Predictions vs. Ground Truth
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], 'r--', linewidth=2)
plt.xlabel("Actual y_test (Ground Truth)")
plt.ylabel("Predicted y_pred (Linear Regression)")
plt.title("Linear Regression: Predictions vs. Ground Truth")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here: